In [ ]:
# Imports
import numpy as np # For storing data and performing computations
SEED = 57131605 # For deterministic testing

In [ ]:
# Parse Data
import email
from email import policy
from pathlib import Path

# Parser
def parse_path(path: Path) -> list[str]:
    out = []
    for file in path.glob('*.txt'):
        with open(file=file, mode='r', encoding='latin-1') as f:
            try:
                message = email.message_from_file(f, policy=policy.default)
                subject = message['subject'] or '' # Get subject message (set to an empty string if not provided)
                body = ""
                if message.is_multipart():
                    for part in message.walk():
                        if part.get_content_type() == 'text/plain':
                            body += part.get_content()
                else:
                    body = message.get_content()
                out.append(subject + " " + body)
            except Exception as e:
                print(f'Failed on {file}: {e}')
                return []
    return out

# Parse folders
spam_emails = parse_path(Path('enron1/spam'))
ham_emails = parse_path(Path('enron1/ham'))

In [ ]:
# Clean data
import re

# Transform every email string into a list of words
def tokenize(text: str) -> list[str]:
    text = text.lower() # Standard formatting
    text = re.sub(r'\d+', ' ', text) # Remove digits
    text = re.sub(r'\W+', ' ', text) # Remove other erroneous characters
    return text.split()

# Tokenize email strings
spam_tokens = [t for t in map(tokenize, spam_emails)]
ham_tokens = [t for t in map(tokenize, ham_emails)]

In [ ]:
# TF-IDF model

# Finding highest frequency counts
word2count = dict()
for lst in spam_tokens + ham_tokens:
    for token in lst:
        if token in word2count:
            word2count[token] += 1
        else:
            word2count[token] = 1

# HYPERPARAMETER: number of words kept -- N_WORDS
N_WORDS = 2000
import heapq
freq_words = set(heapq.nlargest(N_WORDS, word2count, key=word2count.get))

# Truncate Dictionary:
word2count = {w: word2count[w] for w in word2count if w in freq_words}

# Word to Index
WtoI = {w:i for i, w in enumerate(freq_words)}

# Construct word vectors
TF = np.zeros(shape=(5172, 2000), dtype=float)
ys = np.concatenate((np.zeros(shape=(1500,), dtype=int) - 1, np.zeros(shape=(3672,), dtype=int) + 1))

# Construct Bag of Words vectors
for i, tokens in enumerate(spam_tokens + ham_tokens):
    for token in tokens:
        if token in freq_words:
            TF[i, WtoI[token]] += 1 / len(tokens)

IDF = np.log(len(TF) / (np.sum(a=TF, axis=0, keepdims=False) + 1))
xs = TF * IDF

np.random.seed(SEED)
permut = np.random.permutation(len(xs))

xs = xs[permut]
ys = ys[permut]

# Create datasplits
x_train = xs[:4138]
y_train = ys[:4138]
x_test = xs[4138:4655]
y_test = ys[4138:4655]
x_dev = xs[4655:]
y_dev = ys[4655:]

In [ ]:
# Kernel


In [ ]:
# Kernel SVM

# Perform Optimization on Lagrange Multiplier Vector: L
def grad_descent(H, X, Y, C, iters=int(1e3), epsilon=1.0):
    L = np.zeros(shape=(len(X)), dtype=float)
    for _ in range(iters):
        grad = H @ L - 1
        grad = np.clip(grad, -1000, 1000)
        L -= epsilon * grad # Step
        L = np.clip(L, 0, C) # Enforce Non-Negativity
        L = L - Y * (Y @ L) / (Y @ Y + 1e-10) # Project onto Hyperplane to maintain constraint

        if _ % 250 == 0:
            epsilon /= 25
    return L

def train(X, Y, C):
    H = np.outer(Y, Y) * (X @ X.T)
    H /= H.max() # Scale down matrix to control gradient
    H += np.eye(H.shape[0]) * 1e-2 # Reduce eig val spread
    L = grad_descent(H, X, Y, C)

    return L